In [0]:
# ============================================
# PySpark Sales Analytics Pipeline
# Author: Rajendra
# Architecture: Bronze → Silver → Gold
# ============================================

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, avg, count, month, year, round

# In Databricks, SparkSession is already created
spark = spark

print("✅ Spark Session Ready!")
print(f"Spark Version: {spark.version}")

✅ Spark Session Ready!
Spark Version: 4.1.0


In [0]:
# ============================================
# BRONZE LAYER - Load Raw Data
# ============================================

from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

schema = StructType([
    StructField("OrderID", IntegerType(), True),
    StructField("CustomerID", StringType(), True),
    StructField("ProductID", StringType(), True),
    StructField("Quantity", IntegerType(), True),
    StructField("Price", DoubleType(), True),
    StructField("OrderDate", StringType(), True)
])

data = [
    (1001, "C001", "P001", 2, 500.0, "2026-01-01"),
    (1002, "C002", "P002", 1, 1000.0, "2026-01-02"),
    (1003, "C003", "P003", 3, 250.0, "2026-01-03"),
    (1004, "C004", "P004", 2, 750.0, "2026-01-04"),
    (1005, "C005", "P005", 1, 1500.0, "2026-01-05"),
    (1006, "C001", "P002", 4, 1000.0, "2026-01-06"),
    (1007, "C002", "P003", None, 250.0, "2026-01-07"),
    (1008, "C003", "P001", 3, 500.0, "2026-01-08"),
    (1009, None, "P005", 1, 1500.0, "2026-01-09"),
    (1010, "C005", "P004", 2, 750.0, "2026-01-10"),
    (1011, "C001", "P003", 5, 250.0, "2026-02-01"),
    (1012, "C002", "P001", 2, 500.0, "2026-02-02"),
    (1013, "C003", "P002", 1, 1000.0, "2026-02-03"),
    (1014, "C004", "P004", 3, 750.0, "2026-02-04"),
    (1015, "C005", "P005", 2, 1500.0, "2026-02-05"),
]

bronze_df = spark.createDataFrame(data, schema)

print("✅ BRONZE LAYER - Raw Data Loaded!")
print(f"Total rows: {bronze_df.count()}")
bronze_df.show()

✅ BRONZE LAYER - Raw Data Loaded!
Total rows: 15
+-------+----------+---------+--------+------+----------+
|OrderID|CustomerID|ProductID|Quantity| Price| OrderDate|
+-------+----------+---------+--------+------+----------+
|   1001|      C001|     P001|       2| 500.0|2026-01-01|
|   1002|      C002|     P002|       1|1000.0|2026-01-02|
|   1003|      C003|     P003|       3| 250.0|2026-01-03|
|   1004|      C004|     P004|       2| 750.0|2026-01-04|
|   1005|      C005|     P005|       1|1500.0|2026-01-05|
|   1006|      C001|     P002|       4|1000.0|2026-01-06|
|   1007|      C002|     P003|    NULL| 250.0|2026-01-07|
|   1008|      C003|     P001|       3| 500.0|2026-01-08|
|   1009|      NULL|     P005|       1|1500.0|2026-01-09|
|   1010|      C005|     P004|       2| 750.0|2026-01-10|
|   1011|      C001|     P003|       5| 250.0|2026-02-01|
|   1012|      C002|     P001|       2| 500.0|2026-02-02|
|   1013|      C003|     P002|       1|1000.0|2026-02-03|
|   1014|      C004|   

In [0]:
# ============================================
# SILVER LAYER - Clean & Transform Data
# ============================================

from pyspark.sql.functions import to_date, col, round

# 1. Drop rows with NULL values
silver_df = bronze_df.dropna()

# 2. Convert OrderDate from string to date type
silver_df = silver_df.withColumn("OrderDate", to_date(col("OrderDate"), "yyyy-MM-dd"))

# 3. Add TotalAmount column
silver_df = silver_df.withColumn("TotalAmount", round(col("Quantity") * col("Price"), 2))

# 4. Filter out any negative or zero quantities
silver_df = silver_df.filter(col("Quantity") > 0)

print("✅ SILVER LAYER - Data Cleaned & Transformed!")
print(f"Bronze rows: {bronze_df.count()} → Silver rows: {silver_df.count()}")
print(f"Dropped {bronze_df.count() - silver_df.count()} bad rows")
silver_df.show()

✅ SILVER LAYER - Data Cleaned & Transformed!
Bronze rows: 15 → Silver rows: 13
Dropped 2 bad rows
+-------+----------+---------+--------+------+----------+-----------+
|OrderID|CustomerID|ProductID|Quantity| Price| OrderDate|TotalAmount|
+-------+----------+---------+--------+------+----------+-----------+
|   1001|      C001|     P001|       2| 500.0|2026-01-01|     1000.0|
|   1002|      C002|     P002|       1|1000.0|2026-01-02|     1000.0|
|   1003|      C003|     P003|       3| 250.0|2026-01-03|      750.0|
|   1004|      C004|     P004|       2| 750.0|2026-01-04|     1500.0|
|   1005|      C005|     P005|       1|1500.0|2026-01-05|     1500.0|
|   1006|      C001|     P002|       4|1000.0|2026-01-06|     4000.0|
|   1008|      C003|     P001|       3| 500.0|2026-01-08|     1500.0|
|   1010|      C005|     P004|       2| 750.0|2026-01-10|     1500.0|
|   1011|      C001|     P003|       5| 250.0|2026-02-01|     1250.0|
|   1012|      C002|     P001|       2| 500.0|2026-02-02|     

In [0]:
# ============================================
# GOLD LAYER - Business Analytics
# ============================================

from pyspark.sql.functions import sum, avg, count, round, month, year

# 1. Total Sales by Customer
print("📊 Total Sales by Customer:")
customer_sales = silver_df.groupBy("CustomerID").agg(sum("TotalAmount").alias("TotalSales"), count("OrderID").alias("TotalOrders"), round(avg("TotalAmount"), 2).alias("AvgOrderValue")).orderBy("TotalSales", ascending=False)
customer_sales.show()

# 2. Top Products by Revenue
print("📦 Top Products by Revenue:")
product_sales = silver_df.groupBy("ProductID").agg(sum("TotalAmount").alias("TotalRevenue"), sum("Quantity").alias("TotalQuantitySold")).orderBy("TotalRevenue", ascending=False)
product_sales.show()

# 3. Monthly Sales Trend
print("📅 Monthly Sales Trend:")
monthly_sales = silver_df.withColumn("Month", month("OrderDate")).withColumn("Year", year("OrderDate")).groupBy("Year", "Month").agg(round(sum("TotalAmount"), 2).alias("MonthlySales")).orderBy("Year", "Month")
monthly_sales.show()

print("✅ GOLD LAYER - Analytics Complete!")

📊 Total Sales by Customer:
+----------+----------+-----------+-------------+
|CustomerID|TotalSales|TotalOrders|AvgOrderValue|
+----------+----------+-----------+-------------+
|      C001|    6250.0|          3|      2083.33|
|      C005|    6000.0|          3|       2000.0|
|      C004|    3750.0|          2|       1875.0|
|      C003|    3250.0|          3|      1083.33|
|      C002|    2000.0|          2|       1000.0|
+----------+----------+-----------+-------------+

📦 Top Products by Revenue:
+---------+------------+-----------------+
|ProductID|TotalRevenue|TotalQuantitySold|
+---------+------------+-----------------+
|     P002|      6000.0|                6|
|     P004|      5250.0|                7|
|     P005|      4500.0|                3|
|     P001|      3500.0|                7|
|     P003|      2000.0|                8|
+---------+------------+-----------------+

📅 Monthly Sales Trend:
+----+-----+------------+
|Year|Month|MonthlySales|
+----+-----+------------+
|2026|

In [0]:
# Monthly Sales Trend
from pyspark.sql.functions import sum, round, month, year

monthly_sales = silver_df.withColumn("Month", month("OrderDate")).withColumn("Year", year("OrderDate")).groupBy("Year", "Month").agg(round(sum("TotalAmount"), 2).alias("MonthlySales")).orderBy("Year", "Month")

print("📅 Monthly Sales Trend:")
monthly_sales.show()

📅 Monthly Sales Trend:
+----+-----+------------+
|Year|Month|MonthlySales|
+----+-----+------------+
|2026|    1|     12750.0|
|2026|    2|      8500.0|
+----+-----+------------+



In [0]:
# ============================================
# SAVE AS DELTA TABLES
# ============================================

# Save Silver layer
silver_df.write.format("delta").mode("overwrite").saveAsTable("silver_orders")
print("✅ Silver table saved!")

# Save Gold - Customer Sales
customer_sales.write.format("delta").mode("overwrite").saveAsTable("gold_customer_sales")
print("✅ Gold customer sales table saved!")

# Save Gold - Product Sales
product_sales.write.format("delta").mode("overwrite").saveAsTable("gold_product_sales")
print("✅ Gold product sales table saved!")

# Save Gold - Monthly Trend
monthly_sales.write.format("delta").mode("overwrite").saveAsTable("gold_monthly_trend")
print("✅ Gold monthly trend table saved!")

print("\n🎉 All Delta Tables saved successfully!")

✅ Silver table saved!
✅ Gold customer sales table saved!
✅ Gold product sales table saved!
✅ Gold monthly trend table saved!

🎉 All Delta Tables saved successfully!
